# SOXX constituent basket lead information screen

This notebook is the primary research entrypoint for the statically reviewed plan in `plan.json`. The screen compares fixed OLS model B (SOXX first-hour return) with model C (the same input plus the prior-public 30-member basket return less SOXX). It uses native Qlib calendars, bounded `D.features` reads, phase-specific `DatasetH` objects, and a Qlib Recorder.

**Execution requires the separately sealed owner GO.** The 2024 and 2025 market-data reads and fits require the research owner's separate GO after checking the final plan, exact source hashes, implementation identity, and normalized notebook snapshot. No pre-existing Recorder is read. The 2024 models and receipt must be sealed before any 2025 price, label, or weight construction.

The planned target is `log(SOXX open at right-edge 11:01 / open at right-edge 10:33)`, corresponding to 11:00 and 10:32 fills. The 10:30 factor uses completed 12-bar 5-minute constituent prefixes and 60-bar 1-minute SOXX prefixes. A session stays on the native calendar when data are unknown; incomplete baskets are never partially renormalized.

The sole primary gate is positive fitted C gamma, positive 2025 paired daily squared-loss improvement, and a positive fixed one-sided stationary-bootstrap lower bound (63-session mean block, 10,000 replicates, seed 20261014, quantile 0.025/15). A result is a reused-history information screen only.

In [ ]:
from pathlib import Path
import sys

ROOT = Path('/Users/rziiiii/dev/workspace')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.workflow.basket_etf_lead_information import run

# This call rejects missing/stale GO before market IO, fits or Recorder creation.
SUMMARY = run(root=ROOT)
SUMMARY


## Frozen execution sequence

1. Write the attempt journal before source reads, then validate exact plan, design review, implementation and pinned file bytes.
2. Read only 2024 Qlib source calendars and the two bounded price matrices; build 2024 PIT weights from SEC filing metadata and a phase-only native calendar. Preserve unknown calendar rows.
3. Build native Qlib DatasetH inputs, require at least 200 complete rows, fit B and C once, and save source identities, scaler means/stds, coefficients, masks, and hashes in the new Recorder. Finish and read back the sealed freeze receipt.
4. Only after that readback, query 2025 prices and build 2025 PIT weights. Score the unchanged coefficients, write paired losses and the fixed bootstrap to the same Recorder, and close it.
5. Recheck pinned bytes and source identities. Leave unknown rows explicit and report any contract failure without retrying or silently changing the design.

Reusable mechanics live in `src/workflow/basket_etf_lead_information.py`. This static handoff has no numerical result, Recorder ID, or promotion status.

结论：待 root GO 后执行唯一一次冻结信息筛查。尚无本卡数值结果。2025 为复用历史，信息筛查通过也不构成独立验证、费用后收益或生产晋升。
